# 05 — Clean, reject and deduplicate customer events

**Student exercise**

Complete the tasks in order. Each task includes an expected result, three hints, and a check. All sample data and setup code are included in this notebook.

## 1. Spark configuration

Start a fresh kernel. Use Spark 3.5.x / Scala 2.12 and Java 17. Set `STUDENT_ID` to a lowercase name containing letters, numbers or underscores. Set `SPARK_MASTER=local[2]` for a local Spark session.

In [ ]:
import os
import socket
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

master_url = os.environ.get("SPARK_MASTER", f"spark://{socket.gethostname()}:7077")
STUDENT_ID = "student01"
assert STUDENT_ID and STUDENT_ID[0].islower() and all(c in "abcdefghijklmnopqrstuvwxyz0123456789_" for c in STUDENT_ID)

spark = (
    SparkSession.builder
    .appName("Exercise-05")
    .master(master_url)
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "2")
    .config("spark.redaction.regex", "(?i)secret|password|token|access[.]?key|credential")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

## 2. Sample data

Run this cell, inspect the records, and use the supplied variables in the tasks.

In [ ]:
events=spark.createDataFrame([
 (1,' Asha ','ASHA@EXAMPLE.TEST','2026-09-01'),(1,'Asha Rao','asha@example.test','2026-09-02'),
 (2,'Ben','bad-email','2026-09-01'),(3,' Carla ','carla@example.test','2026-09-01'),
 (4,None,'dev@example.test','2026-09-01')],
 'id INT,name STRING,email STRING,event_date STRING')

## 3. Result checks

This small function checks output values and duplicate counts. It is included here so the notebook can run on its own.

In [ ]:
def check_frame(df, columns, expected):
    assert df is not None, "Complete the task before running the check"
    assert set(columns).issubset(df.columns), "Missing output columns"
    def normalize(row):
        return tuple(round(value, 6) if isinstance(value, float) else value for value in row)
    actual = sorted(repr(normalize(row)) for row in df.select(*columns).collect())
    wanted = sorted(repr(normalize(row)) for row in expected)
    assert actual == wanted, f"Expected {wanted}, got {actual}"
    print("PASS:", columns)

## Task 1: Normalize and label quality

Trim name, lowercase/trim email, flag is_valid when name is nonempty and email contains @.

**Expected result:** Three valid event rows, two invalid.

In [ ]:
normalized = None  # TODO: trim, lower, validity condition, coalesce null condition to False

**Hint 1:** This is a teaching email check, not a full email-address validator.

**Hint 2:** Use coalesce(condition, lit(False)) so a missing name is rejected rather than lost.

**Hint 3:** withColumn(...).withColumn("is_valid", F.coalesce(...)).

### Check

In [ ]:
check_frame(normalized,['id','is_valid'],[(1,True),(1,True),(2,False),(3,True),(4,False)])

## Task 2: Split accepted and rejected events

Produce accepted and rejected; verify they account for all source events.

**Expected result:** Accepted + rejected = 5.

In [ ]:
accepted=None
rejected=None  # TODO: complementary filters

**Hint 1:** Filtering only valid events without tracking rejects hides data loss.

**Hint 2:** Use the same non-null flag for both filters.

**Hint 3:** accepted = normalized.filter(...); rejected = normalized.filter(...).

### Check

In [ ]:
check_frame(accepted,['id'],[(1,),(1,),(3,)])
check_frame(rejected,['id'],[(2,),(4,)])
assert accepted.count()+rejected.count()==events.count()

## Task 3: Keep the latest valid event per key

Use event_date descending to retain latest=ID1 Asha Rao and ID3 Carla. Do not choose a random duplicate.

**Expected result:** Two latest valid customer records.

In [ ]:
latest=None  # TODO: row_number over a per-id window

**Hint 1:** dropDuplicates(id) does not express which differing event should win.

**Hint 2:** Window.partitionBy("id").orderBy(event_date descending).

**Hint 3:** Add row_number, keep rn=1, drop rn. Add a tie-breaker for equal event times in production.

### Check

In [ ]:
check_frame(latest,['id','name','email'],[(1,'Asha Rao','asha@example.test'),(3,'Carla','carla@example.test')])

## More practice

Explain distinct versus dropDuplicates; add a tied event and a deterministic event sequence.

## Common mistakes

NOT NULL logic is three-valued. Ties need a business ordering rule. Do not deduplicate before quality checks without considering consequences.

## Finish

Stop Spark before starting another notebook. For write exercises, run setup again before repeating append tasks. S3 data remains available after stopping Spark.

In [ ]:
spark.stop()